In [ ]:
# ===============================================
# 📄 CONVERT TÀI LIỆU → PDF (BỎ QUA VIDEO)
# ===============================================

from google.colab import drive
drive.mount('/content/drive')

# ---- LINK NGUỒN ----
shared_folder_link = "https://drive.google.com/drive/folders/1pPW181RvJnxDrpV4rcq3Vgd6-Ka-xIvg"

# ---- THƯ MỤC ĐÍCH ----
target_folder = "/content/drive/MyDrive/Tai_lieu_PDF"

# ---- CÀI THƯ VIỆN ----
!apt-get install -y libreoffice >/dev/null
!pip install -q --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

import os, io, re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

auth.authenticate_user()
service = build('drive', 'v3')


# ---- HÀM LẤY ID THƯ MỤC ----
def extract_folder_id(link):
    m = re.search(r'/folders/([a-zA-Z0-9_-]+)', link)
    return m.group(1) if m else None

folder_id = extract_folder_id(shared_folder_link)


# ==================================================
# 📌 DANH SÁCH LOẠI FILE BỎ QUA
# ==================================================
SKIP_EXTENSIONS = {
    '.mp4', '.avi', '.mov', '.mkv', '.flv', '.wmv', '.webm', '.m4v', '.mpeg', '.mpg',
    '.mp3', '.wav', '.flac', '.aac', '.ogg', '.wma', '.m4a',
    '.jpg', '.jpeg', '.png', '.gif', '.bmp', '.svg', '.webp', '.ico',
    '.zip', '.rar', '.7z', '.tar', '.gz',
    '.exe', '.dmg', '.app', '.apk'
}

SKIP_MIME_TYPES = {
    'video/mp4', 'video/x-msvideo', 'video/quicktime', 'video/x-matroska',
    'video/x-flv', 'video/x-ms-wmv', 'video/webm', 'video/mpeg',
    'audio/mpeg', 'audio/wav', 'audio/flac', 'audio/aac', 'audio/ogg',
    'image/jpeg', 'image/png', 'image/gif', 'image/bmp', 'image/svg+xml', 'image/webp',
    'application/vnd.google-apps.form',
    'application/vnd.google-apps.site',
    'application/vnd.google-apps.map',
}

GOOGLE_APPS_EXPORTABLE = {
    'application/vnd.google-apps.document',
    'application/vnd.google-apps.spreadsheet',
    'application/vnd.google-apps.presentation',
    'application/vnd.google-apps.drawing',
}

OFFICE_EXTENSIONS = {'.docx', '.doc', '.pptx', '.ppt', '.xlsx', '.xls', '.txt', '.rtf', '.odt', '.odp', '.ods'}

DIRECT_PDF_MIME = {
    'application/pdf',
}

OFFICE_MIME_TYPES = {
    'application/vnd.openxmlformats-officedocument.presentationml.presentation',
    'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
    'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet',
    'application/vnd.ms-powerpoint',
    'application/msword',
    'application/vnd.ms-excel',
    'application/vnd.oasis.opendocument.text',
    'application/vnd.oasis.opendocument.presentation',
    'application/vnd.oasis.opendocument.spreadsheet',
    'text/plain',
    'application/rtf',
}


# ==================================================
# 📌 KIỂM TRA FILE CÓ NÊN XỬ LÝ KHÔNG
# ==================================================
def should_process_file(name, mime):
    name_lower = name.lower()
    if any(name_lower.endswith(ext) for ext in SKIP_EXTENSIONS):
        return False
    if mime in SKIP_MIME_TYPES:
        return False
    return True


# ==================================================
# 📌 LẤY TÊN FILE OUTPUT
# ==================================================
def get_pdf_output_path(dest_folder, name):
    base = name
    for ext in ['.docx', '.doc', '.pptx', '.ppt', '.xlsx', '.xls',
                '.txt', '.rtf', '.odt', '.odp', '.ods', '.pdf']:
        if base.lower().endswith(ext):
            base = base[:-len(ext)]
            break
    return os.path.join(dest_folder, base + ".pdf")


# ==================================================
# 📌 EXPORT GOOGLE APPS FILES → PDF
# ==================================================
def export_google_to_pdf(file_id, pdf_path):
    try:
        request = service.files().export_media(fileId=file_id, mimeType="application/pdf")
        with io.FileIO(pdf_path, "wb") as f:
            downloader = MediaIoBaseDownload(f, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        return True
    except Exception as e:
        print(f"  ⚠️ Lỗi export Google Apps: {e}")
        return False


# ==================================================
# 📌 DOWNLOAD FILE THẬT → CONVERT VỚI LIBREOFFICE
# ==================================================
def download_and_convert_office(file_id, name, dest_folder):
    try:
        temp_path = os.path.join("/content", name)

        request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
        with open(temp_path, "wb") as f:
            downloader = MediaIoBaseDownload(f, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()

        os.makedirs(dest_folder, exist_ok=True)
        !libreoffice --headless --convert-to pdf --outdir "{dest_folder}" "{temp_path}" >/dev/null 2>&1

        os.remove(temp_path)
        return True
    except Exception as e:
        print(f"  ⚠️ Lỗi convert LibreOffice: {e}")
        return False


# ==================================================
# 📌 DOWNLOAD PDF GỐC
# ==================================================
def download_pdf(file_id, pdf_path):
    try:
        request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
        with open(pdf_path, "wb") as f:
            downloader = MediaIoBaseDownload(f, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        return True
    except Exception as e:
        print(f"  ⚠️ Lỗi copy PDF: {e}")
        return False


# ==================================================
# 📌 RESOLVE SHORTCUT → LẤY ID THẬT
# ==================================================
def resolve_shortcut(file_id):
    try:
        meta = service.files().get(
            fileId=file_id,
            fields="id,mimeType,shortcutDetails",
            supportsAllDrives=True
        ).execute()
        if meta.get("mimeType") == "application/vnd.google-apps.shortcut":
            return meta["shortcutDetails"]["targetId"]
        return file_id
    except Exception as e:
        print(f"  ⚠️ Không resolve được shortcut: {e}")
        return file_id


# ==================================================
# 📌 XỬ LÝ 1 FILE
# ==================================================
def process_single_file(fid, name, mime, dest_folder):
    if not should_process_file(name, mime):
        print(f"⏭️  Bỏ qua: {name}  [{mime}]")
        return

    out_pdf = get_pdf_output_path(dest_folder, name)

    if os.path.exists(out_pdf):
        print(f"⏩ Đã có sẵn: {name}")
        return

    # CASE 1: Google Apps files (Docs, Sheets, Slides, Drawings)
    if mime in GOOGLE_APPS_EXPORTABLE:
        print(f"🔄 Google Apps → PDF: {name}")
        if export_google_to_pdf(fid, out_pdf):
            print(f"  ✅ Xong")
        return

    # CASE 2: PDF gốc
    if mime in DIRECT_PDF_MIME or name.lower().endswith('.pdf'):
        print(f"📄 Copy PDF: {name}")
        if download_pdf(fid, out_pdf):
            print(f"  ✅ Xong")
        return

    # CASE 3: Office files thật
    if mime in OFFICE_MIME_TYPES or any(name.lower().endswith(ext) for ext in OFFICE_EXTENSIONS):
        print(f"🔄 Office → PDF: {name}")
        if download_and_convert_office(fid, name, dest_folder):
            print(f"  ✅ Xong")
        return

    # CASE 4: Không rõ loại → log để kiểm tra
    print(f"❓ Không xử lý được: {name}  [{mime}]")


# ==================================================
# 📌 DUYỆT THƯ MỤC (hỗ trợ Shared Drive + Shortcut)
# ==================================================
def process_folder(folder_id, dest_folder):
    # Resolve nếu bản thân folder này là shortcut
    folder_id = resolve_shortcut(folder_id)

    query = f"'{folder_id}' in parents and trashed=false"
    results = service.files().list(
        q=query,
        fields="files(id,name,mimeType,shortcutDetails)",
        pageSize=1000,
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ).execute()
    files = results.get("files", [])

    for file in files:
        fid  = file["id"]
        name = file["name"]
        mime = file["mimeType"]

        # ---- SHORTCUT ----
        if mime == "application/vnd.google-apps.shortcut":
            try:
                target_id   = file["shortcutDetails"]["targetId"]
                target_mime = file["shortcutDetails"]["targetMimeType"]
            except KeyError:
                print(f"⚠️  Shortcut lỗi (không có targetId): {name}")
                continue

            if target_mime == "application/vnd.google-apps.folder":
                new_folder = os.path.join(dest_folder, name)
                os.makedirs(new_folder, exist_ok=True)
                print(f"\n📁 Shortcut → Folder: {name}/")
                process_folder(target_id, new_folder)
            else:
                process_single_file(target_id, name, target_mime, dest_folder)
            continue

        # ---- THƯ MỤC THƯỜNG ----
        if mime == "application/vnd.google-apps.folder":
            new_folder = os.path.join(dest_folder, name)
            os.makedirs(new_folder, exist_ok=True)
            print(f"\n📁 Đang vào thư mục: {name}/")
            process_folder(fid, new_folder)
            continue

        # ---- FILE THƯỜNG ----
        process_single_file(fid, name, mime, dest_folder)


# ==================================================
# CHẠY
# ==================================================
os.makedirs(target_folder, exist_ok=True)
print("🔍 Đang xử lý tài liệu (bỏ qua video, audio, image)...\n")
process_folder(folder_id, target_folder)
print("\n🎉 Hoàn tất! Kiểm tra thư mục:", target_folder)